In [1]:
"""
04_dataset_selection.py

Select the model sample from the cleaned dataset:
- keep BV/BVBA legal forms, accounting schemas 1/2/7, bookyears 2017-2019
- build the 1-year-ahead bankruptcy target
- drop columns that are almost entirely missing within this sample
- save dataset_selected.csv
"""


'\n04_dataset_selection.py\n\nSelect the model sample from the cleaned dataset:\n- keep BV/BVBA legal forms, accounting schemas 1/2/7, bookyears 2017-2019\n- build the 1-year-ahead bankruptcy target\n- drop columns that are almost entirely missing within this sample\n- save dataset_selected.csv\n'

In [2]:
from utils.load_data_cleaned import load_data_cleaned

df = load_data_cleaned()


In [3]:
import pandas as pd

from config import (
    BOOKYEARS, DATASET_SELECTED, LOGS, MISSING_THRESHOLD, NATURES,
    RECHTSVORMEN, TARGET,
)
from utils.print_section import print_section

df = df[df["rechtsvorm"].isin(RECHTSVORMEN)]

# ------------------------------------------------------------------
# Descriptives on the BV/BVBA sample, before further filtering
# ------------------------------------------------------------------
print_section("Basic shape")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

print_section("Unique companies")
print(f"Unique VAT numbers: {df['vat'].nunique():,}")

print_section("Panel structure")
obs_per_company = df.groupby("vat").size()
print(obs_per_company.describe())

print_section("Bookyears")
print(df["bookyear"].value_counts().sort_index())

print_section("Legal forms")
print(df["rechtsvorm"].value_counts(dropna=False).sort_index())

print_section("Schema types")
print(df["nature"].value_counts(dropna=False).sort_index())

print_section("Industries")
print(df["industry"].value_counts(dropna=False).sort_index())

print_section("Bankruptcies")
failed_companies = df.loc[df["jaar_van_faling"].notna(), "vat"].nunique()
total_companies = df["vat"].nunique()
print(f"Unique failed companies: {failed_companies:,}")
print(f"Failure rate: {failed_companies / total_companies:.2%}")

print_section("Bankruptcy years")
print(df["jaar_van_faling"].value_counts(dropna=False).sort_index())

# ------------------------------------------------------------------
# t0 -> t1 availability: how many observations per bookyear actually
# have a bankruptcy outcome recorded for the following year.
#
# NOTE: the fail rate drops from 0.35% (2017) to 0.32% (2018) to
# 0.19% (2019). Part of that is real: Belgium suspended most
# bankruptcy filings for several months in 2020 as a COVID relief
# measure, so "failure in 2020" is structurally under-recorded.
# Keep this in mind when interpreting model performance on the 2019
# bookyear (see the train/test split below).
# ------------------------------------------------------------------
print_section("T0 -> T1 availability")

summary = pd.DataFrame([
    {
        "bookyear": year,
        "observations": (df["bookyear"] == year).sum(),
        "fail_next_year": ((df["bookyear"] == year) & (df["jaar_van_faling"] == year + 1)).sum(),
    }
    for year in sorted(df["bookyear"].unique())
])
summary["fail_rate_pct"] = (summary["fail_next_year"] / summary["observations"] * 100).round(3)
print(summary)

# ------------------------------------------------------------------
# Apply sample selection criteria and build the target
# ------------------------------------------------------------------
df = df[df["nature"].isin(NATURES)]
df = df[df["bookyear"].isin(BOOKYEARS)]
df[TARGET] = (df["jaar_van_faling"] == df["bookyear"] + 1).astype(int)

print_section("Target distribution")
print(df[TARGET].value_counts())
print()
print((df[TARGET].value_counts(normalize=True) * 100).round(3))

# ------------------------------------------------------------------
# Drop columns that became almost entirely missing after filtering
# down to this sample.
# ------------------------------------------------------------------
print_section("Remove high-missing columns after sample selection")

PROTECTED_COLUMNS = ["jaar_van_faling", "fail", "faling_datum", TARGET]

missing_pct = df.isna().mean()
drop_cols = [
    col for col in missing_pct[missing_pct >= MISSING_THRESHOLD].index
    if col not in PROTECTED_COLUMNS
]
print(f"Columns to remove: {len(drop_cols)}")
for col in sorted(drop_cols):
    print(col)

original_rows, original_columns = len(df), len(df.columns)
original_companies = df["vat"].nunique()

df = df.drop(columns=drop_cols)
print(f"\nDataset shape after column removal: {df.shape}")

# ------------------------------------------------------------------
# Save the selected dataset
# ------------------------------------------------------------------
df.to_csv(DATASET_SELECTED, index=False)
print(f"Selected dataset saved to: {DATASET_SELECTED}")

# ------------------------------------------------------------------
# Log
# ------------------------------------------------------------------
print_section("Write drop log")

LOG_PATH = LOGS / "dataset_selection_log.txt"
target_counts = df[TARGET].value_counts()
target_pct = df[TARGET].value_counts(normalize=True).mul(100).round(3)

with open(LOG_PATH, "w") as f:
    f.write("DATASET SELECTION LOG\n")
    f.write("=" * 60 + "\n\n")

    f.write("Selection criteria\n")
    f.write("-" * 60 + "\n")
    f.write(f"rechtsvorm: {', '.join(RECHTSVORMEN)}\n")
    f.write(f"nature: {', '.join(str(n) for n in NATURES)}\n")
    f.write(f"bookyear: {', '.join(str(y) for y in BOOKYEARS)}\n")
    f.write("target: jaar_van_faling == bookyear + 1\n\n")

    f.write(f"Original dataset: {original_rows:,} rows x {original_columns:,} columns\n")
    f.write(f"Final dataset: {len(df):,} rows x {len(df.columns):,} columns\n")
    f.write(f"Unique companies: {original_companies:,}\n\n")

    f.write("Target distribution\n")
    f.write("-" * 60 + "\n")
    for target_value in sorted(target_counts.index):
        f.write(f"{target_value}: {target_counts[target_value]:,} ({target_pct[target_value]:.3f}%)\n")
    f.write("\n")

    f.write(f"High-missing columns removed ({len(drop_cols)})\n")
    f.write("-" * 60 + "\n")
    for col in sorted(drop_cols):
        f.write(f"{col}\n")

print(f"Log written to: {LOG_PATH}")



Basic shape
Rows: 2,550,039
Columns: 106

Unique companies
Unique VAT numbers: 373,680

Panel structure
count    373680.000000
mean          6.824125
std           2.839130
min           1.000000
25%           5.000000
50%           9.000000
75%           9.000000
max           9.000000
dtype: float64

Bookyears
bookyear
2010     22092
2011    203387
2012    228592
2013    245907
2014    266684
2015    285973
2016    303456
2017    320353
2018    336357
2019    325365
2020     11873
Name: count, dtype: int64

Legal forms
rechtsvorm
BV       162008
BVBA    2388031
Name: count, dtype: int64

Schema types
nature
1     1721472
2       36411
3           9
4          15
5           2
7      791759
81        157
82         21
87        193
Name: count, dtype: int64

Industries
industry
0.0      61900
10.0     38762
11.0     47145
12.0     47488
13.0     29007
14.0    886132
15.0    192715
16.0    488786
17.0    403189
18.0    273830
19.0     81085
Name: count, dtype: int64

Bankruptcies
Uniq


Target distribution
target
0    978996
1      2822
Name: count, dtype: int64

target
0    99.713
1     0.287
Name: proportion, dtype: float64

Remove high-missing columns after sample selection


Columns to remove: 16
openingfaling
rub11
rub12
rub15
rub21_28
rub29
rub290
rub291
rub37
rub439
rub51_53
rub66
rub70
rub76
rub9900
rub9902

Dataset shape after column removal: (981818, 91)


Selected dataset saved to: /Users/seymaciftci/Code/bankruptcy-master-thesis/data/processed/dataset_selected.csv

Write drop log
Log written to: /Users/seymaciftci/Code/bankruptcy-master-thesis/logs/dataset_selection_log.txt
